### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

### Loading Data

In [2]:
emissions_df = pd.read_csv('historical_emissions.csv')
emissions_df.head(3)

,Country,Data source,Sector,Gas,Unit,2018,2017,2016,2015,2014,...,1999,1998,1997,1996,1995,1994,1993,1992,1991,1990
0,World,CAIT,Total including LUCF,CO2,MtCO₂e,36441.55,35588.70,35160.60,34521.91,34558.59,...,24957.30,24895.32,25292.92,24214.92,23890.22,23260.29,23124.00,22988.29,23056.40,22849.92
1,China,CAIT,Total including LUCF,CO2,MtCO₂e,9663.36,9367.67,9164.21,9120.27,9184.77,...,2799.84,2882.75,2779.27,2715.50,2735.48,2414.50,2294.12,2068.77,1952.78,1823.96
2,United States,CAIT,Total including LUCF,CO2,MtCO₂e,4749.57,4581.90,4656.84,4563.52,4683.35,...,5191.66,5172.06,5129.29,4864.46,4708.31,4654.52,4581.76,4461.62,4389.50,4426.40


In [3]:
# keep_default_na=False so Namibia's ISO-2 code "NA" isn't read as a missing value
country_code = pd.read_csv('country_code.csv', keep_default_na=False, na_values=[''])
country_code.head(3)

,Unnamed: 0,Country_name,code_2digit,code_3digit
0,2,Afghanistan,AF,AFG
1,3,Aland Islands,AX,ALA
2,4,Albania,AL,ALB


In [4]:
country_df = pd.read_csv(
    'world_country_and_usa_states_latitude_and_longitude_values.csv',
    keep_default_na=False, na_values=['']
)
country_df.head()

,country_code,latitude,longitude,country,usa_state_code,usa_state_latitude,usa_state_longitude,usa_state
0,AD,42.546245,1.601554,Andorra,AK,63.588753,-154.493062,Alaska
1,AE,23.424076,53.847818,United Arab Emirates,AL,32.318231,-86.902298,Alabama
2,AF,33.939110,67.709953,Afghanistan,AR,35.201050,-91.831833,Arkansas
3,AG,17.060816,-61.796428,Antigua and Barbuda,AZ,34.048928,-111.093731,Arizona
4,AI,18.220554,-63.068615,Anguilla,CA,36.778261,-119.417932,California


### Checking the match (fixed)

**Bug in the original notebook:** `right_on='Country_name'` was used against
`country_df`, but `country_df`'s country-name column is actually called
`country` (lowercase) — `Country_name` only exists on `country_code`, a
different dataframe. That mismatch meant the merge key didn't exist, so the
join wasn't doing what it looked like it was doing.

Checking against the correct column (`country_df['country']`) below.

In [5]:
emissions_df['Match_found'] = emissions_df['Country'].isin(country_df['country'])
emissions_df['Match_found'].value_counts()

Match_found
True     186
False      9
Name: count, dtype: int64

In [6]:
# Method 1
unmatched = emissions_df.loc[~emissions_df['Match_found'], 'Country'].tolist()
print(f"{len(unmatched)} unmatched countries:")
unmatched

# Method 2
# emissions_df[~emissions_df['Match_found']]

9 unmatched countries:


['World',
 'European Union (27)',
 'Democratic Republic of the Congo',
 'Myanmar',
 'Republic of Congo',
 'South Sudan',
 'Macedonia',
 'Eswatini',
 'Sao Tome and Principe']

### Fixing the remaining mismatches with an alias map

The leftover mismatches are real naming differences between the datasets
(e.g. "South Korea" vs "Korea (South)", "Vietnam" vs "Viet Nam", "Russia" vs
"Russian Federation"). A plain `isin`/merge can never catch these — built an
explicit alias map instead, found by manually cross-checking names across
all three files.

`World` and `European Union (27)` are aggregates, not countries, so they're
expected to stay unmatched (no ISO code, no coordinates).

In [7]:
ALIASES_TO_COUNTRY_DF = {
    'democratic republic of the congo': 'congo [drc]',
    'republic of congo': 'congo [republic]',
    'eswatini': 'swaziland',
    'macedonia': 'macedonia [fyrom]',
    'myanmar': 'myanmar [burma]',
}

ALIASES_TO_COUNTRY_CODE = {
    'brunei': 'brunei darussalam',
    'iran': 'iran, islamic republic of',
    'macedonia': 'macedonia, republic of',
    'micronesia': 'micronesia, federated states of',
    'north korea': 'korea (north)',
    'south korea': 'korea (south)',
    'republic of congo': 'congo (brazzaville)',
    'democratic republic of the congo': 'congo, (kinshasa)',
    'saint vincent and the grenadines': 'saint vincent and grenadines',
    'syria': 'syrian arab republic (syria)',
    'tanzania': 'tanzania, united republic of',
    'united states': 'united states of america',
    'venezuela': 'venezuela (bolivarian republic)',
    'vietnam': 'viet nam',
    'eswatini': 'swaziland',
    'russia': 'russian federation',
    'laos': 'lao pdr',
}

def norm(s):
    return s.strip().lower()

emissions_df['_key'] = emissions_df['Country'].map(norm)
country_df['_key'] = country_df['country'].map(norm)
country_code['_key'] = country_code['Country_name'].map(norm)

emissions_df['_key_geo'] = emissions_df['_key'].map(lambda k: ALIASES_TO_COUNTRY_DF.get(k, k))
emissions_df['_key_code'] = emissions_df['_key'].map(lambda k: ALIASES_TO_COUNTRY_CODE.get(k, k))

### Merging in ISO codes and lat/long

In [8]:
# Step 1: bring in ISO 2/3 codes from country_code.csv
code_lookup = country_code[['_key', 'code_2digit', 'code_3digit']].rename(
    columns={'_key': '_lookup_key', 'code_2digit': 'iso2', 'code_3digit': 'iso3'}
)
merged_df = emissions_df.merge(
    code_lookup, left_on='_key_code', right_on='_lookup_key', how='left'
).drop(columns=['_lookup_key'])

# Step 2: bring in latitude/longitude from country_df
geo_lookup = country_df[['_key', 'latitude', 'longitude']].rename(columns={'_key': '_lookup_key'})
merged_df = merged_df.merge(
    geo_lookup, left_on='_key_geo', right_on='_lookup_key', how='left'
).drop(columns=['_lookup_key'])

# clean up helper columns
merged_df = merged_df.drop(columns=['_key', '_key_geo', '_key_code', 'Match_found'])
merged_df.head()

,Country,Data source,Sector,Gas,Unit,2018,2017,2016,2015,2014,...,1995,1994,1993,1992,1991,1990,iso2,iso3,latitude,longitude
0,World,CAIT,Total including LUCF,CO2,MtCO₂e,36441.55,35588.70,35160.60,34521.91,34558.59,...,23890.22,23260.29,23124.00,22988.29,23056.40,22849.92,NaN,NaN,NaN,NaN
1,China,CAIT,Total including LUCF,CO2,MtCO₂e,9663.36,9367.67,9164.21,9120.27,9184.77,...,2735.48,2414.50,2294.12,2068.77,1952.78,1823.96,CN,CHN,35.861660,104.195397
2,United States,CAIT,Total including LUCF,CO2,MtCO₂e,4749.57,4581.90,4656.84,4563.52,4683.35,...,4708.31,4654.52,4581.76,4461.62,4389.50,4426.40,US,USA,37.090240,-95.712891
3,European Union (27),CAIT,Total including LUCF,CO2,MtCO₂e,2636.99,2692.12,2669.54,2321.61,2263.78,...,3113.68,3060.09,3072.40,3133.06,3247.48,3286.44,NaN,NaN,NaN,NaN
4,India,CAIT,Total including LUCF,CO2,MtCO₂e,2400.25,2267.16,2149.01,2085.38,2072.03,...,519.98,466.79,431.31,409.09,386.17,341.32,IN,IND,20.593684,78.962880


### Verifying match quality

In [9]:
known_aggregates = {'world', 'european union (27)'}

still_unmatched_iso = merged_df[
    merged_df['iso2'].isna() & ~merged_df['Country'].str.lower().isin(known_aggregates)
]
still_unmatched_geo = merged_df[
    merged_df['latitude'].isna() & ~merged_df['Country'].str.lower().isin(known_aggregates)
]

print("Unmatched ISO codes (excl. World / EU):", still_unmatched_iso['Country'].tolist())
print("Unmatched lat/long (excl. World / EU):", still_unmatched_geo['Country'].tolist())
# South Sudan and Sao Tome and Principe are genuinely missing from the lat/long
# source file -- not a naming issue, just absent data.

Unmatched ISO codes (excl. World / EU): []
Unmatched lat/long (excl. World / EU): ['South Sudan', 'Sao Tome and Principe']


### Final tidy-up and save

In [14]:
year_cols = [c for c in merged_df.columns if c.isdigit()]
front_cols = ['Country', 'iso2', 'iso3', 'latitude', 'longitude']
merged_df = merged_df[front_cols + year_cols]

merged_df.to_csv('merged_country_co2_geo.csv', index=False)
merged_df[merged_df['latitude'].isna()]

,Country,iso2,iso3,latitude,longitude,2018,2017,2016,2015,2014,...,1999,1998,1997,1996,1995,1994,1993,1992,1991,1990
0,World,NaN,NaN,NaN,NaN,36441.55,35588.70,35160.60,34521.91,34558.59,...,24957.30,24895.32,25292.92,24214.92,23890.22,23260.29,23124.00,22988.29,23056.40,22849.92
3,European Union (27),NaN,NaN,NaN,NaN,2636.99,2692.12,2669.54,2321.61,2263.78,...,3079.46,3134.03,3142.23,3210.93,3113.68,3060.09,3072.40,3133.06,3247.48,3286.44
126,South Sudan,SS,SSD,NaN,NaN,10.28,10.44,10.67,8.51,8.07,...,7.06,7.03,7.02,7.01,7.01,6.94,6.94,6.94,6.94,6.94
174,Sao Tome and Principe,ST,STP,NaN,NaN,0.35,0.35,0.34,0.34,0.33,...,0.06,0.06,0.06,0.06,0.06,0.06,0.05,0.05,0.06,0.06


In [13]:
merged_df

,Country,iso2,iso3,latitude,longitude,2018,2017,2016,2015,2014,...,1999,1998,1997,1996,1995,1994,1993,1992,1991,1990
0,World,NaN,NaN,NaN,NaN,36441.55,35588.70,35160.60,34521.91,34558.59,...,24957.30,24895.32,25292.92,24214.92,23890.22,23260.29,23124.00,22988.29,23056.40,22849.92
1,China,CN,CHN,35.861660,104.195397,9663.36,9367.67,9164.21,9120.27,9184.77,...,2799.84,2882.75,2779.27,2715.50,2735.48,2414.50,2294.12,2068.77,1952.78,1823.96
2,United States,US,USA,37.090240,-95.712891,4749.57,4581.90,4656.84,4563.52,4683.35,...,5191.66,5172.06,5129.29,4864.46,4708.31,4654.52,4581.76,4461.62,4389.50,4426.40
3,European Union (27),NaN,NaN,NaN,NaN,2636.99,2692.12,2669.54,2321.61,2263.78,...,3079.46,3134.03,3142.23,3210.93,3113.68,3060.09,3072.40,3133.06,3247.48,3286.44
4,India,IN,IND,20.593684,78.962880,2400.25,2267.16,2149.01,2085.38,2072.03,...,683.00,618.73,600.38,555.60,519.98,466.79,431.31,409.09,386.17,341.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,Nauru,NR,NRU,-0.522778,166.931503,0.07,0.07,0.06,0.05,0.05,...,0.09,0.10,0.10,0.10,0.11,0.11,0.11,0.12,0.13,0.13
191,Tuvalu,TV,TUV,-7.109535,177.649330,0.01,0.01,0.01,0.01,0.01,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.00
192,Niue,NU,NIU,-19.054445,-169.867233,0.00,0.00,0.00,0.00,0.00,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
193,Fiji,FJ,FJI,-16.578193,179.414413,-0.65,-0.73,-0.81,-1.03,-1.24,...,-1.74,-1.76,-1.74,-1.72,-1.78,-1.79,-1.79,-1.78,-1.81,-1.81
